# danh sách nghề IT đã gom lại

In [1]:
import pandas as pd
import os

# ===== 1. Load 2 files =====
core = pd.read_csv(
    "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/04_esco_it_core_occupations.csv"
)

ext = pd.read_csv(
    "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/05_esco_it_extended_occupations.csv"
)

# ===== 2. Gắn nhãn group =====
core["group"] = "core"
ext["group"] = "extended"

# ===== 3. Gộp lại =====
df_all = pd.concat([core, ext], ignore_index=True)

# ===== 4. Làm sạch =====
df_all["preferredLabel"] = (
    df_all["preferredLabel"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_all = df_all[df_all["preferredLabel"] != ""].copy()

# ===== 5. Bỏ trùng nếu có =====
df_all = df_all.drop_duplicates(subset=["preferredLabel", "group"]).copy()

# ===== 6. Xem nhanh =====
print("Total IT occupations:", len(df_all))
print(df_all["group"].value_counts())

display(
    df_all[["preferredLabel", "group"]]
    .sort_values(["group", "preferredLabel"])
    .reset_index(drop=True)
)

Total IT occupations: 54
group
core        46
extended     8
Name: count, dtype: int64


,preferredLabel,group
0,ICT application developer,core
1,ICT network administrator,core
2,ICT network engineer,core
3,ICT system administrator,core
4,ICT system developer,core
5,IoT developer,core
6,artificial intelligence engineer,core
7,blockchain developer,core
8,business developer,core
9,business intelligence manager,core


# Export file tổng

In [2]:
output_dir = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "06_esco_it_all_occupations.csv")
df_all.to_csv(output_path, index=False)

print(f"Saved merged occupations to: {output_path}")

Saved merged occupations to: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/notebook/ESCO_taxonomy/notebook_clean/06_esco_it_all_occupations.csv


# Check Overlap

* Tác dụng của check overlap

    - Giữ cho 2 nhóm tách bạch

        Nếu đã mất công chia:

        - IT kỹ thuật lõi

        - IT mở rộng
        
        thì ideally một nghề nên nằm 1 nhóm thôi.

    - Phát hiện lỗi do rule lọc vì:

        Có khi:

        - keyword của core bắt được một nghề

        - keyword của extended cũng bắt được chính nghề đó

    - Giúp quyết định nghề nào thuộc nhóm nào “chính thức”

        Nếu overlap xảy ra, phải chốt: nghề này để ở core hay để ở extended

        chứ không nên để cả 2 nếu muốn bộ taxonomy sạch.

* Ví dụ dễ hiểu

    Giả sử kết quả là:

    Overlap count: 2
    
        ['business intelligence manager', 'systems analyst']

    thì nghĩa là:

        2 nghề này đang bị gán ở cả core lẫn extended

    giờ phải xem:
    
        - systems analyst nên nằm ở đâu?
        
        - business intelligence manager nên nằm ở đâu?

    Nếu overlap = 0 thì sao?

        Thì đẹp.

    Nghĩa là:

    - core và extended đã tách nhau rõ
    
    - không có nghề trùng nhóm
    
    - file tổng của  sạch hơn

In [3]:
core_set = set(core["preferredLabel"].dropna().astype(str).str.strip().str.lower())
ext_set = set(ext["preferredLabel"].dropna().astype(str).str.strip().str.lower())

overlap = sorted(core_set & ext_set)

print("Overlap count:", len(overlap))
print(overlap)

Overlap count: 0
[]
